# Enhanced Streamlit Data Lineage App

This notebook contains an enhanced Streamlit application to visualize data lineage from your Excel file with interactive network graphs.

In [ ]:
!pip install streamlit pandas openpyxl pyvis networkx

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import networkx as nx
from pyvis.network import Network
import streamlit.components.v1 as components
import os

st.set_page_config(page_title="Data Lineage Explorer", layout="wide")

st.title("📂 Data Lineage Explorer")

file_path = r"C:\Users\samar\Downloads\Data_Lineage 1.xlsx"

def load_data(path):
    if not os.path.exists(path):
        st.error(f"File not found at {path}")
        return None
    df = pd.read_excel(path)
    df.columns = df.columns.str.strip()  # This line fixes the error
    return df


df = load_data(file_path)

if df is not None:
    # Sidebar Filters
    st.sidebar.header("🔍 Configuration")
    
    # Filters
    source_systems = sorted(df['Source System'].unique().tolist())
    selected_sources = st.sidebar.multiselect("Source Systems", source_systems, default=source_systems)
    
    target_systems = sorted(df['Target System'].unique().tolist())
    selected_targets = st.sidebar.multiselect("Target Systems", target_systems, default=target_systems)
    
    # Filter logic
    mask = df['Source System'].isin(selected_sources) & df['Target System'].isin(selected_targets)
    filtered_df = df[mask]
    
    # Tabs for different views
    tab1, tab2, tab3 = st.tabs(["📈 Dashboard", "🔗 Lineage Graph", "📄 Raw Data"])
    
    with tab1:
        st.subheader("Data Overview")
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Source Tables", len(filtered_df['Source Table'].unique()))
        c2.metric("Target Tables", len(filtered_df['Target Table'].unique()))
        c3.metric("Total Mappings", len(filtered_df))
        c4.metric("Complex Transformations", len(filtered_df[filtered_df['Transformation Business Logic'].notna()]))

        st.divider()
        st.write("### Source to Target Mapping Counts")
        mapping_counts = filtered_df.groupby(['Source System', 'Target System']).size().reset_index(name='Count')
        st.bar_chart(mapping_counts, x='Source System', y='Count', color='Target System')

    with tab2:
        st.subheader("Interactive Lineage Graph")
        st.info("This graph shows the flow of data from Source Tables to Target Tables.")
        
        # Build NetworkX Graph
        G = nx.DiGraph()
        
        for _, row in filtered_df.iterrows():
            source_node = f"{row['Source System']}\n{row['Source Table']}"
            target_node = f"{row['Target System']}\n{row['Target Table']}"
            
            G.add_node(source_node, title=f"System: {row['Source System']}", color='#97c2fc')
            G.add_node(target_node, title=f"System: {row['Target System']}", color='#eb82a9')
            G.add_edge(source_node, target_node, label=row['Source Column'] + ' -> ' + row['Target Column'])
            
        # Create PyVis Network
        net = Network(height='600px', width='100%', bgcolor='#ffffff', font_color='black', directed=True)
        net.from_nx(G)
        
        # Custom Physics settings for better layout
        net.set_options("""
        var options = {
          "physics": {
            "forceAtlas2Based": {
              "gravitationalConstant": -50,
              "centralGravity": 0.01,
              "springLength": 100,
              "springConstant": 0.08
            },
            "maxVelocity": 50,
            "solver": "forceAtlas2Based",
            "timestep": 0.35,
            "stabilization": { "iterations": 150 }
          }
        }
        """)
        
        net.save_graph("lineage_graph.html")
        
        # Display the graph
        HtmlFile = open("lineage_graph.html", 'r', encoding='utf-8')
        source_code = HtmlFile.read() 
        components.html(source_code, height=650)

    with tab3:
        st.subheader("Mapping Details")
        st.dataframe(filtered_df, use_container_width=True)


### Step 3: Run the app
Execute the cell below to start the Streamlit server.

In [ ]:
!streamlit run app.py